### **WL vs FRL PBS distribution in FRL tomograms**

This script tests for self-affinity in the dsitribution of WL vs FRL PBS while:

1. excluding within-row neighbors (because those are trivial along the same filament/row), and
2. restricting neighbors to the same membrane surface via a ψ-gating proxy (because membranes are curved).

So the workflow is:

1. Build a between-row neighbor graph at the particle level, within each tomogram.
2. For each particle, compute the fraction of its valid between-row neighbors that are the same type.
3. Summarize that “same-type neighbor fraction” separately for WL and FRL.
4. Use a row-label permutation within each tomogram to get the null distribution that respects WL rarity and row structure.

##### Imports & function definitions

In [ ]:
import argparse
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree

In [ ]:
# ---------- function definitions ----------
def read_data_particles_star(path):
    with open(path) as f:
        lines = f.readlines()
    dp_idx = next(i for i, L in enumerate(lines) if L.strip().lower() == "data_particles")
    loop_idx = next(j for j in range(dp_idx + 1, len(lines)) if lines[j].strip().lower() == "loop_")
    cols = []
    for k in range(loop_idx + 1, len(lines)):
        txt = lines[k].strip()
        if txt.startswith("_rln"):
            name = txt.split()[0]
            if name not in cols:
                cols.append(name)
            continue
        if txt and (txt[0].isdigit() or txt[0] in "+-"):
            data_start = k
            break
    data = []
    for row in lines[data_start:]:
        row = row.strip()
        if not row or row.startswith("#") or row.startswith("_rln"):
            continue
        parts = row.split(None, len(cols) - 1)
        if len(parts) != len(cols):
            continue
        data.append(parts)
    df = pd.DataFrame(data, columns=cols)
    return df, lines, dp_idx, loop_idx, data_start, cols

# ---------- config ----------
TOMO_COL = "_rlnTomoName"
ROW_COL  = "_rlnRowID"
TYPE_COL = "_rlnParticleType"

COORD_COLS = [
    "_rlnCenteredCoordinateXAngst",
    "_rlnCenteredCoordinateYAngst",
    "_rlnCenteredCoordinateZAngst",
]
ANGLE_COLS = ["_rlnAngleRot", "_rlnAngleTilt", "_rlnAnglePsi"]

def _wrap_angle_deg(x):
    return ((x + 180.0) % 360.0) - 180.0

def _coerce_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def load_star_as_type(path, ptype):
    df, *_ = read_data_particles_star(path)
    df = df.copy()
    df[TYPE_COL] = ptype
    df["__source_star"] = str(path)

    # strip tomo name
    if TOMO_COL not in df.columns:
        raise ValueError(f"Missing required column {TOMO_COL} in {path}")
    df[TOMO_COL] = df[TOMO_COL].astype(str).str.strip()

    # require RowID
    if ROW_COL not in df.columns:
        raise ValueError(f"Missing required column {ROW_COL} in {path}")

    # numeric coercion
    num_cols = COORD_COLS + ANGLE_COLS + [
        ROW_COL,
        "_rlnLCCmax",
        "_rlnCutOff",
        "_rlnSearchStd",
        "_rlnTomoTiltSeriesPixelSize",
        "_rlnRandomSubset",
    ]##
    df = _coerce_numeric(df, num_cols)

    # require coords + tomo + row
    missing_coords = [c for c in COORD_COLS if c not in df.columns]
    if missing_coords:
        raise ValueError(f"Missing required coordinate columns in {path}: {missing_coords}")

    df = df.dropna(subset=COORD_COLS + [TOMO_COL, ROW_COL]).copy()
    df[ROW_COL] = df[ROW_COL].astype(int)

    # wrap angles (if present)
    for c in ANGLE_COLS:
        if c in df.columns:
            df[c] = _wrap_angle_deg(df[c].astype(float))

    return df

def drop_singlets(df):
    df = df.copy()
    df[ROW_COL] = pd.to_numeric(df[ROW_COL], errors="coerce")
    df = df[df[ROW_COL].notna()].copy()
    df[ROW_COL] = df[ROW_COL].astype(int)
    return df[df[ROW_COL] != -1].copy()

def enforce_global_unique_rowids_by_offset(wl, frl):
    """
    Makes ROW_COL globally unique across merged WL+FRL by offsetting FRL row IDs
    by (max WL row ID + 1).

    Correct sanity check:
      - allows many particles per row (expected duplicates in ROW_COL at particle level)
      - verifies that WL row IDs are not reused across multiple tomograms (and same for FRL)
        i.e. a numeric RowID corresponds to a single row object in the whole list.

    If RowIDs are reused across tomograms, use the factorize-based function instead.
    """
    wl = wl.copy()
    frl = frl.copy()

    wl[ROW_COL] = pd.to_numeric(wl[ROW_COL], errors="coerce").astype("Int64")
    frl[ROW_COL] = pd.to_numeric(frl[ROW_COL], errors="coerce").astype("Int64")
    wl = wl[wl[ROW_COL].notna()].copy()
    frl = frl[frl[ROW_COL].notna()].copy()
    wl[ROW_COL] = wl[ROW_COL].astype(int)
    frl[ROW_COL] = frl[ROW_COL].astype(int)

    # keep originals
    wl[ROW_COL + "_orig"]  = wl[ROW_COL]
    frl[ROW_COL + "_orig"] = frl[ROW_COL]

    # --- sanity: does a numeric RowID appear in multiple tomos? ---
    def _rowid_multi_tomo(df, label):
        row_objs = df[[TOMO_COL, ROW_COL]].drop_duplicates()
        counts = row_objs.groupby(ROW_COL)[TOMO_COL].nunique()
        multi = counts[counts > 1]
        if len(multi):
            ex = (row_objs[row_objs[ROW_COL].isin(multi.index[:10])]
                  .sort_values([ROW_COL, TOMO_COL]))
            raise ValueError(
                f"{label}: {len(multi)} RowID values appear in >1 tomogram. "
                f"Offsetting would collide logically. Use factorize-based global RowIDs.\n"
                f"Examples (first few repeated RowIDs and their tomos):\n{ex.head(30)}"
            )

    _rowid_multi_tomo(wl, "WL")
    _rowid_multi_tomo(frl, "FRL")

    # --- offset FRL row IDs to be disjoint from WL row IDs ---
    offset = (int(wl[ROW_COL].max()) + 1) if len(wl) else 0
    frl[ROW_COL] = frl[ROW_COL] + offset

    merged = pd.concat([wl, frl], ignore_index=True)
    merged["_particleID"] = np.arange(len(merged), dtype=np.int64)

    # final sanity: row IDs should now be disjoint between WL and FRL
    if merged[ROW_COL].isna().any():
        raise ValueError("Unexpected NaNs in RowID after merge.")

    return merged

def enforce_global_unique_rowids_by_factorize(wl, frl):
    wl = wl.copy(); frl = frl.copy()
    wl[ROW_COL + "_orig"]  = wl[ROW_COL].astype(int)
    frl[ROW_COL + "_orig"] = frl[ROW_COL].astype(int)
    merged = pd.concat([wl, frl], ignore_index=True)

    key = (
        merged[TOMO_COL].astype(str).str.strip()
        + "|" + merged[TYPE_COL].astype(str).str.strip()
        + "|" + merged[ROW_COL + "_orig"].astype(str)
    )
    merged[ROW_COL], _ = pd.factorize(key, sort=True)
    merged["_particleID"] = np.arange(len(merged), dtype=np.int64)
    return merged


def keep_only_tomos_with_both_types(df):
    tomo_has_both = (
        df.groupby(TOMO_COL)[TYPE_COL]
          .apply(lambda s: {"WL", "FRL"}.issubset(set(s.astype(str))))
    )
    keep = tomo_has_both[tomo_has_both].index
    return df[df[TOMO_COL].isin(keep)].copy()

def summarize(df, title=""):
    if title:
        print(title)
    print("particles:", len(df))
    print("tomos:", df[TOMO_COL].nunique())
    print("types:", df[TYPE_COL].value_counts().to_dict())
    print("rows:", df[ROW_COL].nunique())
    print()

# === Analysis helpers ===

# --- column names (match your df) ---
TOMO_COL = "_rlnTomoName"
ROW_COL  = "_rlnRowID"
TYPE_COL = "_rlnParticleType"

XCOL = "_rlnCenteredCoordinateXAngst"
YCOL = "_rlnCenteredCoordinateYAngst"
ZCOL = "_rlnCenteredCoordinateZAngst"
PSI_COL = "_rlnAnglePsi"

def angdiff_deg(a, b):
    """Absolute wrapped angle difference (degrees), supports scalar a and array b."""
    return np.abs(((b - a + 180.0) % 360.0) - 180.0)

def encode_types_01(df):
    """
    WL -> 0, FRL -> 1
    """
    t = df[TYPE_COL].astype(str).to_numpy()
    out = np.empty(len(t), dtype=np.int8)
    out[t == "WL"]  = 0
    out[t == "FRL"] = 1
    bad = ~np.isin(t, ["WL", "FRL"])
    if bad.any():
        raise ValueError(f"Unexpected types: {pd.Series(t[bad]).value_counts().to_dict()}")
    return out

def build_between_row_neighbors_psi_gated(
    df,
    r_max=2500.0,        # Å
    dpsi_max=30.0,       # degrees
    k_keep=1,            # neighbors to keep per particle after filtering
    k_query=80,          # initial KDTree kNN to consider before filtering
):
    """
    For each tomogram separately:
      - query k_query nearest neighbors in 3D
      - filter to within r_max, different row, and |Δpsi|<=dpsi_max
      - keep up to k_keep (closest) neighbors

    Returns:
      neighbors: (N, k_keep) int32, global particle indices, -1 if missing
      distances: (N, k_keep) float, Å, NaN if missing
    """
    if cKDTree is None:
        raise ImportError("scipy not available (needed for cKDTree). Install scipy or load a module providing it.")

    N = len(df)
    neighbors = -np.ones((N, k_keep), dtype=np.int32)
    distances = np.full((N, k_keep), np.nan, dtype=float)

    tomo = df[TOMO_COL].astype(str).to_numpy()
    row  = df[ROW_COL].to_numpy(int)
    psi  = df[PSI_COL].to_numpy(float)
    xyz  = df[[XCOL, YCOL, ZCOL]].to_numpy(float)

    for t in np.unique(tomo):
        idx_g = np.where(tomo == t)[0]
        if idx_g.size < 2:
            continue

        xyz_t = xyz[idx_g]
        row_t = row[idx_g]
        psi_t = psi[idx_g]

        tree = cKDTree(xyz_t)

        k = min(k_query + 1, idx_g.size)  # +1 includes self
        d, j = tree.query(xyz_t, k=k)     # (n,k)

        # ensure 2D shape even if k==1
        if d.ndim == 1:
            d = d[:, None]
            j = j[:, None]

        for a in range(idx_g.size):
            cand = j[a, 1:]
            cand_d = d[a, 1:]

            # radius gate (optional)
            if r_max is not None:
                m = cand_d <= r_max
                cand = cand[m]
                cand_d = cand_d[m]
                if cand.size == 0:
                    continue

            # exclude same row
            m = row_t[cand] != row_t[a]
            cand = cand[m]
            cand_d = cand_d[m]
            if cand.size == 0:
                continue

            # psi gate
            m = angdiff_deg(psi_t[a], psi_t[cand]) <= dpsi_max
            cand = cand[m]
            cand_d = cand_d[m]
            if cand.size == 0:
                continue

            take = min(k_keep, cand.size)
            neighbors[idx_g[a], :take] = idx_g[cand[:take]]
            distances[idx_g[a], :take] = cand_d[:take]

    return neighbors, distances

def particle_same_type_fraction(labels01, neighbors):
    """
    labels01: (N,) int {0,1}
    neighbors: (N,k) int, -1 where missing
    Returns:
      frac_same: (N,) float in [0,1] or NaN if no valid neighbor
      n_valid: (N,) int number of valid neighbors
    """
    valid = neighbors != -1
    n_valid = valid.sum(axis=1).astype(np.int32)

    nbr = neighbors.copy()
    nbr[~valid] = 0  # safe index
    same = (labels01[nbr] == labels01[:, None]) & valid

    frac = np.full(len(labels01), np.nan, dtype=float)
    m = n_valid > 0
    frac[m] = same.sum(axis=1)[m] / n_valid[m]
    return frac, n_valid

def summarize_homophily(labels01, frac_same):
    wl = labels01 == 0
    frl = labels01 == 1
    out = {
        "S_WL":  float(np.nanmean(frac_same[wl])),
        "S_FRL": float(np.nanmean(frac_same[frl])),
    }
    out["delta_WL_minus_FRL"] = out["S_WL"] - out["S_FRL"]
    return out

def prepare_row_permutation(df, labels01):
    """
    Precompute structures to permute row labels within each tomogram
    without redoing neighbor search.
    """
    tomo = df[TOMO_COL].astype(str).to_numpy()
    row_ids = df[ROW_COL].to_numpy(int)

    uniq_rows, row_idx = np.unique(row_ids, return_inverse=True)
    n_rows = uniq_rows.size

    # row label from first encountered particle; verify consistency
    row_label0 = np.full(n_rows, -1, dtype=np.int8)
    mixed = 0
    for i in range(len(df)):
        r = row_idx[i]
        lab = labels01[i]
        if row_label0[r] == -1:
            row_label0[r] = lab
        elif row_label0[r] != lab:
            mixed += 1
    if mixed:
        raise ValueError(
            f"Found {mixed} particles whose row contains both WL and FRL labels. "
            "If this is expected, we should switch to particle-label permutation instead."
        )

    # rows per tomogram
    rows_by_tomo = {}
    for t in np.unique(tomo):
        rows_by_tomo[t] = np.unique(row_idx[tomo == t])

    return row_idx, row_label0, rows_by_tomo

def permutation_test_rowlabels(
    neighbors,
    row_idx_per_particle,
    row_label0,
    rows_by_tomo,
    n_perm=5000,
    seed=0,
):
    """
    Permute row labels WITHIN each tomogram (preserves WL/FRL row counts per tomogram),
    then evaluate S_WL, S_FRL, delta.
    """
    rng = np.random.default_rng(seed)

    N = len(row_idx_per_particle)
    stats = np.zeros((n_perm, 3), dtype=float)  # S_WL, S_FRL, delta

    for p in range(n_perm):
        row_lab = row_label0.copy()
        for t, rows in rows_by_tomo.items():
            row_lab[rows] = rng.permutation(row_label0[rows])

        lab_p = row_lab[row_idx_per_particle]  # (N,)
        frac_p, _ = particle_same_type_fraction(lab_p, neighbors)
        s = summarize_homophily(lab_p, frac_p)
        stats[p, 0] = s["S_WL"]
        stats[p, 1] = s["S_FRL"]
        stats[p, 2] = s["delta_WL_minus_FRL"]

    return stats

def drop_frl_overlapping_wl(df, radius=60.0):
    """
    Within each tomogram, drop FRL particles within `radius` (Å) of any WL
    particle (duplicate picks: FRL template occasionally picks WL PBS).
    Returns deduplicated df + count dropped.
    """
    xyz = df[[XCOL, YCOL, ZCOL]].to_numpy(float)
    tomo = df[TOMO_COL].astype(str).to_numpy()
    is_wl = df[TYPE_COL].astype(str).to_numpy() == "WL"

    drop_mask = np.zeros(len(df), dtype=bool)
    for t in np.unique(tomo):
        in_t = tomo == t
        wl_i  = np.where(in_t & is_wl)[0]
        frl_i = np.where(in_t & ~is_wl)[0]
        if wl_i.size == 0 or frl_i.size == 0:
            continue
        tree = cKDTree(xyz[wl_i])
        hit = tree.query_ball_point(xyz[frl_i], r=radius)
        drop_mask[frl_i[[len(h) > 0 for h in hit]]] = True

    n_dropped = int(drop_mask.sum())
    return df[~drop_mask].copy(), n_dropped

#### Import data

In [ ]:
wl_star = "wl-PBS_rowID.star"
frl_star = "frl-PBS_rowID.star"

In [ ]:
wl  = load_star_as_type(wl_star, "WL")
frl = load_star_as_type(frl_star, "FRL")

wl  = drop_singlets(wl)
frl = drop_singlets(frl)


df = enforce_global_unique_rowids_by_offset(wl, frl)

df, n_dup = drop_frl_overlapping_wl(df, radius=60.0)
print(f"Dropped {n_dup} FRL duplicate picks (<=60 Å from a WL pick)")

df = keep_only_tomos_with_both_types(df)
summarize(df, "Merged (tomos with both WL+FRL)")


#### Analysis

In [ ]:
# === Cell: build psi-gated between-row neighbors ===

# parameters to tune
r_max    = 2500.0   # Å: max 3D distance to consider
dpsi_max = 50     # deg: psi similarity gate
k_keep   = 3        # keep 1 nearest between-row neighbor per particle (start here)
k_query  = 45       # initial kNN candidates per particle

labels01 = encode_types_01(df)

neighbors, distances = build_between_row_neighbors_psi_gated(
    df,
    r_max=r_max,
    dpsi_max=dpsi_max,
    k_keep=k_keep,
    k_query=k_query,
)

frac_obs, n_valid = particle_same_type_fraction(labels01, neighbors)
obs = summarize_homophily(labels01, frac_obs)

print("Valid neighbor fraction:", float((n_valid > 0).mean()))
print(obs)


In [ ]:
# build row-permutation null for the current neighbor graph
row_idx_per_particle, row_label0, rows_by_tomo = prepare_row_permutation(df, labels01)

perm_stats = permutation_test_rowlabels(
    neighbors,
    row_idx_per_particle,
    row_label0,
    rows_by_tomo,
    n_perm=5000,
    seed=0,
)

# Observed: probability WL's between-row neighbor is WL (given your neighbor definition)
obs_S_WL = float(np.nanmean(frac_obs[labels01 == 0]))   # WL coded as 0

# Null distribution from row-label permutations
null_S_WL = perm_stats[:, 0]

# One-sided p-value for "WL is more next to WL than expected"
p_one = (np.sum(null_S_WL >= obs_S_WL) + 1) / (len(null_S_WL) + 1)

# Effect sizes
null_mean = float(null_S_WL.mean())
null_sd   = float(null_S_WL.std(ddof=1))
z = (obs_S_WL - null_mean) / null_sd
ci = np.quantile(null_S_WL, [0.025, 0.975])

enrichment_ratio = obs_S_WL / null_mean if null_mean != 0 else np.nan
enrichment_diff  = obs_S_WL - null_mean

print("Observed S_WL:", obs_S_WL)
print("Null mean ± sd:", null_mean, null_sd)
print("Null 95% CI:", (float(ci[0]), float(ci[1])))
print("Enrichment (diff):", enrichment_diff)
print("Enrichment (ratio):", enrichment_ratio)
print("z:", float(z))
print("p(one-sided, WL more WL-neigh):", float(p_one))


**Interpretation:** WL particles are ~3× more likely to have a WL nearest between-row neighbor on the same membrane (as defined by your ψ gate) than expected if WL/FRL row identities were randomly assigned. This supports WL–WL clustering / affinity at the row–row neighborhood scale captured by your neighbor definition.

#### Sanity check:

##### Functions

In [ ]:
# === Add this to your hidden cell (robustness + sanity checks toolkit) ===

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree
import matplotlib.pyplot as plt

# --- columns (match your df) ---
TOMO_COL = "_rlnTomoName"
ROW_COL  = "_rlnRowID"
TYPE_COL = "_rlnParticleType"

XCOL = "_rlnCenteredCoordinateXAngst"
YCOL = "_rlnCenteredCoordinateYAngst"
ZCOL = "_rlnCenteredCoordinateZAngst"

ROT_COL  = "_rlnAngleRot"
TILT_COL = "_rlnAngleTilt"
PSI_COL  = "_rlnAnglePsi"

def angdiff_deg(a, b):
    return np.abs(((b - a + 180.0) % 360.0) - 180.0)

def encode_types_01(df):
    t = df[TYPE_COL].astype(str).to_numpy()
    out = np.empty(len(t), dtype=np.int8)
    out[t == "WL"]  = 0
    out[t == "FRL"] = 1
    bad = ~np.isin(t, ["WL", "FRL"])
    if bad.any():
        raise ValueError(f"Unexpected types: {pd.Series(t[bad]).value_counts().to_dict()}")
    return out

def precompute_knn_by_tomo(df, k_query=80):
    """
    Precompute per-particle kNN candidates within each tomogram (in 3D).
    Returns:
      knn_idx (N,k_query) global indices, -1 where missing
      knn_dist (N,k_query) distances (Å), NaN where missing
    """
    xyz = df[[XCOL, YCOL, ZCOL]].to_numpy(float)
    tomo = df[TOMO_COL].astype(str).to_numpy()

    N = len(df)
    knn_idx  = -np.ones((N, k_query), dtype=np.int32)
    knn_dist = np.full((N, k_query), np.nan, dtype=float)

    for t in np.unique(tomo):
        idx_g = np.where(tomo == t)[0]
        if idx_g.size < 2:
            continue

        xyz_t = xyz[idx_g]
        tree = cKDTree(xyz_t)

        k = min(k_query + 1, idx_g.size)  # +1 includes self
        d, j = tree.query(xyz_t, k=k)

        if d.ndim == 1:
            d = d[:, None]
            j = j[:, None]

        # drop self-hit; keep up to k_query
        d = d[:, 1:]
        j = j[:, 1:]

        take = min(k_query, j.shape[1])
        knn_dist[idx_g, :take] = d[:, :take]

        # convert local -> global indices
        knn_idx[idx_g, :take] = idx_g[j[:, :take]]

    return knn_idx, knn_dist

def build_neighbors_from_knn(
    df,
    knn_idx,
    knn_dist,
    r_max=2500.0,
    k_keep=1,
    dpsi_max=None,
    normal_gate=None,   # dict like {"theta_max_deg": 25.0, "axis": "z"}
):
    """
    Build final neighbors from precomputed candidates with filters:
      - within r_max
      - different row
      - optional psi gate: |Δpsi| <= dpsi_max
      - optional normal gate: dot(n_i, n_j) >= cos(theta_max)
    """
    N = len(df)
    row = df[ROW_COL].to_numpy(int)

    neighbors = -np.ones((N, k_keep), dtype=np.int32)
    distances = np.full((N, k_keep), np.nan, dtype=float)

    psi = None
    if dpsi_max is not None:
        psi = df[PSI_COL].to_numpy(float)

    normals = None
    cos_th = None
    if normal_gate is not None:
        theta = float(normal_gate.get("theta_max_deg", 25.0))
        cos_th = np.cos(np.deg2rad(theta))
        axis = normal_gate.get("axis", "z")
        normals = particle_axis_in_tomo(df, axis=axis)  # (N,3)

    for i in range(N):
        cand = knn_idx[i]
        cand_d = knn_dist[i]

        m = cand != -1
        cand = cand[m]
        cand_d = cand_d[m]

        if cand.size == 0:
            continue

        if r_max is not None:
            m = cand_d <= r_max
            cand = cand[m]
            cand_d = cand_d[m]
            if cand.size == 0:
                continue

        # exclude same row
        m = row[cand] != row[i]
        cand = cand[m]
        cand_d = cand_d[m]
        if cand.size == 0:
            continue

        # psi gate
        if dpsi_max is not None:
            m = angdiff_deg(psi[i], psi[cand]) <= dpsi_max
            cand = cand[m]
            cand_d = cand_d[m]
            if cand.size == 0:
                continue

        # normal gate
        if normal_gate is not None:
            dots = (normals[cand] * normals[i]).sum(axis=1)
            m = dots >= cos_th
            cand = cand[m]
            cand_d = cand_d[m]
            if cand.size == 0:
                continue

        take = min(k_keep, cand.size)
        neighbors[i, :take] = cand[:take]
        distances[i, :take] = cand_d[:take]

    return neighbors, distances

def particle_same_type_fraction(labels01, neighbors):
    valid = neighbors != -1
    n_valid = valid.sum(axis=1).astype(np.int32)

    nbr = neighbors.copy()
    nbr[~valid] = 0
    same = (labels01[nbr] == labels01[:, None]) & valid

    frac = np.full(len(labels01), np.nan, dtype=float)
    m = n_valid > 0
    frac[m] = same.sum(axis=1)[m] / n_valid[m]
    return frac, n_valid

def summarize_S_WL(labels01, frac_same):
    return float(np.nanmean(frac_same[labels01 == 0]))

def check_row_purity(df):
    # rows should not mix WL and FRL
    g = df.groupby(ROW_COL)[TYPE_COL].nunique()
    mixed = g[g > 1]
    return mixed

def prepare_row_permutation(df, labels01):
    row_ids = df[ROW_COL].to_numpy(int)
    tomo = df[TOMO_COL].astype(str).to_numpy()

    uniq_rows, row_idx = np.unique(row_ids, return_inverse=True)
    n_rows = uniq_rows.size

    row_label0 = np.full(n_rows, -1, dtype=np.int8)
    mixed = 0
    for i in range(len(df)):
        r = row_idx[i]
        lab = labels01[i]
        if row_label0[r] == -1:
            row_label0[r] = lab
        elif row_label0[r] != lab:
            mixed += 1
    if mixed:
        raise ValueError(
            f"Rows are not pure types ({mixed} particles contradict row label). "
            "Fix row IDs or switch to particle-label permutation."
        )

    rows_by_tomo = {}
    for t in np.unique(tomo):
        rows_by_tomo[t] = np.unique(row_idx[tomo == t])

    return row_idx, row_label0, rows_by_tomo

def permute_rowlabels_null_S_WL(df, neighbors, labels01, n_perm=5000, seed=0):
    row_idx_per_particle, row_label0, rows_by_tomo = prepare_row_permutation(df, labels01)
    rng = np.random.default_rng(seed)

    null = np.zeros(n_perm, dtype=float)
    for p in range(n_perm):
        row_lab = row_label0.copy()
        for t, rows in rows_by_tomo.items():
            row_lab[rows] = rng.permutation(row_label0[rows])

        lab_p = row_lab[row_idx_per_particle]
        frac_p, _ = particle_same_type_fraction(lab_p, neighbors)
        null[p] = summarize_S_WL(lab_p, frac_p)
    return null

def permute_particlelabels_null_S_WL(df, neighbors, labels01, n_perm=5000, seed=0):
    """
    Shuffle particle labels within each tomogram (preserves WL count per tomo).
    """
    tomo = df[TOMO_COL].astype(str).to_numpy()
    rng = np.random.default_rng(seed)

    null = np.zeros(n_perm, dtype=float)
    for p in range(n_perm):
        lab_p = labels01.copy()
        for t in np.unique(tomo):
            idx = np.where(tomo == t)[0]
            lab_p[idx] = rng.permutation(lab_p[idx])

        frac_p, _ = particle_same_type_fraction(lab_p, neighbors)
        null[p] = summarize_S_WL(lab_p, frac_p)
    return null

def particle_axis_in_tomo(df, axis="z"):
    """
    Map a chosen local axis (x/y/z) into tomogram coordinates using ZYZ Euler angles.
    This is a geometric gate alternative to ψ.
    """
    rot  = np.deg2rad(df[ROT_COL].to_numpy(float))
    tilt = np.deg2rad(df[TILT_COL].to_numpy(float))
    psi  = np.deg2rad(df[PSI_COL].to_numpy(float))

    # local axis vector
    if axis == "z":
        v = np.array([0.0, 0.0, 1.0])
    elif axis == "y":
        v = np.array([0.0, 1.0, 0.0])
    elif axis == "x":
        v = np.array([1.0, 0.0, 0.0])
    else:
        raise ValueError("axis must be 'x', 'y', or 'z'")

    # Build R = Rz(rot) @ Ry(tilt) @ Rz(psi), apply to v, vectorized
    cr, sr = np.cos(rot), np.sin(rot)
    ct, st = np.cos(tilt), np.sin(tilt)
    cp, sp = np.cos(psi), np.sin(psi)

    # Rz(psi) @ v
    vx1 =  cp*v[0] - sp*v[1]
    vy1 =  sp*v[0] + cp*v[1]
    vz1 =  v[2]

    # Ry(tilt) @ (..)
    vx2 =  ct*vx1 + st*vz1
    vy2 =  vy1
    vz2 = -st*vx1 + ct*vz1

    # Rz(rot) @ (..)
    vx3 = cr*vx2 - sr*vy2
    vy3 = sr*vx2 + cr*vy2
    vz3 = vz2

    n = np.stack([vx3, vy3, vz3], axis=1)
    # normalize
    n /= np.linalg.norm(n, axis=1, keepdims=True)
    return n

def row_weighted_S_WL(df, neighbors, labels01):
    """
    Row-weighted version of S_WL:
      - for each WL row, compute fraction of WL particles whose neighbor is WL
      - average across WL rows equally (not weighting long rows more)
    """
    valid = neighbors[:, 0] != -1  # assumes k_keep>=1; use nearest neighbor only
    nbr0 = neighbors[:, 0].copy()
    nbr0[~valid] = 0
    same = (labels01[nbr0] == labels01) & valid

    # only WL particles contribute to WL-row scores
    wl_mask = labels01 == 0
    rows = df[ROW_COL].to_numpy(int)

    # per-row mean over WL particles in that row with a valid neighbor
    row_scores = {}
    for r in np.unique(rows[wl_mask]):
        idx = np.where((rows == r) & wl_mask & valid)[0]
        if idx.size == 0:
            continue
        row_scores[r] = same[idx].mean()

    if len(row_scores) == 0:
        return np.nan

    return float(np.mean(list(row_scores.values())))

def plot_neighbor_edges(df, neighbors, tomo_name, r_max=None, max_particles=4000, max_edges=3000):
    tomo_arr = df[TOMO_COL].astype(str).to_numpy()
    type_arr = df[TYPE_COL].astype(str).to_numpy()
    x_arr = df[XCOL].to_numpy(float)
    y_arr = df[YCOL].to_numpy(float)
    z_arr = df[ZCOL].to_numpy(float)

    tomo_name = str(tomo_name)
    idx_pos = np.where(tomo_arr == tomo_name)[0]
    if idx_pos.size == 0:
        raise ValueError(f"No particles for tomo: {tomo_name}")

    if idx_pos.size > max_particles:
        idx_pos = np.random.default_rng(0).choice(idx_pos, size=max_particles, replace=False)

    sampled = np.zeros(len(df), dtype=bool)
    sampled[idx_pos] = True

    plt.figure(figsize=(7, 7))
    m_wl = type_arr[idx_pos] == "WL"
    plt.scatter(x_arr[idx_pos][m_wl],  y_arr[idx_pos][m_wl],  s=5, label="WL", color='lightseagreen')
    plt.scatter(x_arr[idx_pos][~m_wl], y_arr[idx_pos][~m_wl], s=5, label="FRL", color='crimson')

    drawn = 0
    for i in idx_pos:
        j = neighbors[i, 0]
        if j == -1:
            continue
        if tomo_arr[j] != tomo_name:
            continue
        if not sampled[j]:
            continue

        d = ((x_arr[i]-x_arr[j])**2 + (y_arr[i]-y_arr[j])**2 + (z_arr[i]-z_arr[j])**2) ** 0.5
        if (r_max is not None) and (d > r_max):
            continue

        plt.plot([x_arr[i], x_arr[j]], [y_arr[i], y_arr[j]], linewidth=0.5, alpha=0.3)
        drawn += 1
        if drawn >= max_edges:
            break

    plt.title(f"{tomo_name} (XY projection, {drawn} edges shown)")
    plt.legend()
    plt.axis("equal")
    plt.show()

def drop_frl_overlapping_wl(df, radius=60.0):
    """
    Within each tomogram, drop FRL particles within `radius` (Å) of any WL
    particle (duplicate picks: FRL template occasionally picks WL PBS).
    Returns deduplicated df + count dropped.
    """
    xyz = df[[XCOL, YCOL, ZCOL]].to_numpy(float)
    tomo = df[TOMO_COL].astype(str).to_numpy()
    is_wl = df[TYPE_COL].astype(str).to_numpy() == "WL"

    drop_mask = np.zeros(len(df), dtype=bool)
    for t in np.unique(tomo):
        in_t = tomo == t
        wl_i  = np.where(in_t & is_wl)[0]
        frl_i = np.where(in_t & ~is_wl)[0]
        if wl_i.size == 0 or frl_i.size == 0:
            continue
        tree = cKDTree(xyz[wl_i])
        hit = tree.query_ball_point(xyz[frl_i], r=radius)
        drop_mask[frl_i[[len(h) > 0 for h in hit]]] = True

    n_dropped = int(drop_mask.sum())
    return df[~drop_mask].copy(), n_dropped

##### Checks

In [ ]:
# === Cell 1: basic sanity diagnostics (ψ-gate selectivity + neighbor availability) ===

labels01 = encode_types_01(df)

# 1) Are rows pure type?
mixed = check_row_purity(df)
print("Mixed-type rows:", int(len(mixed)))
if len(mixed):
    display(mixed.head(20))

# 2) ψ distributions by type
display(df.groupby(TYPE_COL)[PSI_COL].describe()[["mean","std","min","max"]])

# 3) Precompute kNN candidates once
k_query = 80
knn_idx, knn_dist = precompute_knn_by_tomo(df, k_query=k_query)
print("kNN precomputed:", knn_idx.shape, knn_dist.shape)


In [ ]:
# === Cell 2: pick a baseline neighbor definition and recompute observed S_WL + row-permutation null ===

r_max    = 25000.0
dpsi_max = 30.0
k_keep   = 3

neighbors, distances = build_neighbors_from_knn(
    df, knn_idx, knn_dist,
    r_max=r_max, k_keep=k_keep,
    dpsi_max=dpsi_max,
    normal_gate=None,
)

frac_obs, n_valid = particle_same_type_fraction(labels01, neighbors)
obs_S_WL = summarize_S_WL(labels01, frac_obs)

print("Valid neighbor fraction by type:")
display(pd.DataFrame({"type": df[TYPE_COL], "has_nbr": (n_valid>0)}).groupby("type")["has_nbr"].mean())

print("Observed S_WL:", obs_S_WL)

# row-label permutation null (your main null)
n_perm = 5000
null_row = permute_rowlabels_null_S_WL(df, neighbors, labels01, n_perm=n_perm, seed=0)

p_one = (np.sum(null_row >= obs_S_WL) + 1) / (len(null_row) + 1)
ci = np.quantile(null_row, [0.025, 0.975])

print("Null (row-label) mean ± sd:", float(null_row.mean()), float(null_row.std(ddof=1)))
print("Null 95% CI:", (float(ci[0]), float(ci[1])))
print("p(one-sided):", float(p_one))
print("enrichment ratio:", float(obs_S_WL / null_row.mean()))


In [ ]:
# === Cell 3: robustness sweep over (r_max, dpsi_max, k_keep) using the precomputed kNN ===

def run_config(r_max, dpsi_max, k_keep, n_perm=2000, seed=0):
    neighbors, _ = build_neighbors_from_knn(
        df, knn_idx, knn_dist,
        r_max=r_max, k_keep=k_keep,
        dpsi_max=dpsi_max,
        normal_gate=None,
    )
    frac, n_valid = particle_same_type_fraction(labels01, neighbors)
    obs = summarize_S_WL(labels01, frac)
    null = permute_rowlabels_null_S_WL(df, neighbors, labels01, n_perm=n_perm, seed=seed)
    p_one = (np.sum(null >= obs) + 1) / (len(null) + 1)
    return {
        "r_max": r_max, "dpsi_max": dpsi_max, "k_keep": k_keep,
        "valid_frac": float((n_valid>0).mean()),
        "obs_S_WL": float(obs),
        "null_mean": float(null.mean()),
        "enrich_ratio": float(obs / null.mean()),
        "p_one": float(p_one),
    }

grid_r    = [1500.0, 2500.0, 3500.0]
grid_dpsi = [15.0, 25.0, 35.0, 45.0, 90]
grid_k    = [1, 3, 5, 10]

rows = []
for r in grid_r:
    for dp in grid_dpsi:
        for kk in grid_k:
            rows.append(run_config(r, dp, kk, n_perm=2000, seed=0))

sweep = pd.DataFrame(rows).sort_values(["p_one","enrich_ratio"], ascending=[True, False])
display(sweep.head(40))


In [ ]:
# === Cell 4: row-weighted S_WL (does the effect survive without long rows dominating?) ===

# requires k_keep >= 1; uses nearest neighbor only
roww_obs = row_weighted_S_WL(df, neighbors, labels01)
print("Row-weighted observed S_WL:", roww_obs)

# row-weighted null via row-label permutations
row_idx_per_particle, row_label0, rows_by_tomo = prepare_row_permutation(df, labels01)

rng = np.random.default_rng(0)
n_perm = 5000
null_roww = np.zeros(n_perm, dtype=float)

for p in range(n_perm):
    row_lab = row_label0.copy()
    for t, rows_ in rows_by_tomo.items():
        row_lab[rows_] = rng.permutation(row_label0[rows_])
    lab_p = row_lab[row_idx_per_particle]
    null_roww[p] = row_weighted_S_WL(df, neighbors, lab_p)

p_one = (np.sum(null_roww >= roww_obs) + 1) / (len(null_roww) + 1)
ci = np.quantile(null_roww, [0.025, 0.975])
print("Row-weighted null mean ± sd:", float(np.nanmean(null_roww)), float(np.nanstd(null_roww, ddof=1)))
print("Row-weighted null 95% CI:", (float(ci[0]), float(ci[1])))
print("p(one-sided):", float(p_one))
print("enrichment ratio:", float(roww_obs / np.nanmean(null_roww)))


In [ ]:
# === Cell 5: alternative null (particle-label shuffle within tomogram) ===

n_perm = 5000
null_part = permute_particlelabels_null_S_WL(df, neighbors, labels01, n_perm=n_perm, seed=0)

p_one = (np.sum(null_part >= obs_S_WL) + 1) / (len(null_part) + 1)
ci = np.quantile(null_part, [0.025, 0.975])

print("Particle-shuffle null mean ± sd:", float(null_part.mean()), float(null_part.std(ddof=1)))
print("Particle-shuffle null 95% CI:", (float(ci[0]), float(ci[1])))
print("p(one-sided):", float(p_one))
print("enrichment ratio:", float(obs_S_WL / null_part.mean()))


In [ ]:
# === Cell 6: normal-vector gate instead of ψ-gate (recommended cross-check) ===
# Choose axis='z' (common) unless you know the membrane normal is another particle axis.

r_max = 2500.0
k_keep = 1
normal_gate = {"theta_max_deg": 25.0, "axis": "z"}  # try 15, 25, 35 deg

neighbors_n, _ = build_neighbors_from_knn(
    df, knn_idx, knn_dist,
    r_max=r_max, k_keep=k_keep,
    dpsi_max=None,
    normal_gate=normal_gate,
)

frac_n, n_valid_n = particle_same_type_fraction(labels01, neighbors_n)
obs_S_WL_n = summarize_S_WL(labels01, frac_n)
print("Observed S_WL (normal gate):", obs_S_WL_n)
print("Valid neighbor fraction:", float((n_valid_n>0).mean()))

null_row_n = permute_rowlabels_null_S_WL(df, neighbors_n, labels01, n_perm=5000, seed=0)
p_one_n = (np.sum(null_row_n >= obs_S_WL_n) + 1) / (len(null_row_n) + 1)
print("Null mean:", float(null_row_n.mean()), "p(one-sided):", float(p_one_n), "ratio:", float(obs_S_WL_n / null_row_n.mean()))


In [ ]:
# === Cell 7: visual spot-check (pick a tomo) ===
df = df.reset_index(drop=True).copy()
df["_particleID"] = np.arange(len(df), dtype=np.int64)

labels01 = encode_types_01(df)

knn_idx, knn_dist = precompute_knn_by_tomo(df, k_query=80)
neighbors, distances = build_neighbors_from_knn(
    df, knn_idx, knn_dist,
    r_max=2500.0,
    k_keep=10,
    dpsi_max=90.0,
    normal_gate=None,
)

# pick by unique tomogram index, not particle-row index
tomo_list = df[TOMO_COL].drop_duplicates().to_list()

tomo_idx = 0
tomo_name = tomo_list[tomo_idx]

print("Plotting:", tomo_name)

plot_neighbor_edges(df, neighbors, tomo_name, max_particles=4000, max_edges=6000)

In [ ]:
# === FRL/WL particle ratio per tomogram ===

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional: exclude singletons if they are still present
df_ratio = df[df[ROW_COL] != -1].copy()

# Count WL and FRL particles per tomogram
counts = (
    df_ratio
    .groupby([TOMO_COL, TYPE_COL])
    .size()
    .unstack(fill_value=0)
)

# Make sure both columns exist
for col in ["WL", "FRL"]:
    if col not in counts.columns:
        counts[col] = 0

counts = counts[["WL", "FRL"]].copy()

# Calculate ratio
counts["FRL_WL_ratio"] = counts["WL"] / counts["FRL"].replace(0, np.nan)

# Summary statistics across tomograms
mean_ratio = counts["FRL_WL_ratio"].mean()
std_ratio  = counts["FRL_WL_ratio"].std()
sem_ratio  = counts["FRL_WL_ratio"].sem()
n_tomos    = counts["FRL_WL_ratio"].notna().sum()

print(f"mean WL fraction = {mean_ratio:.3f}")
print(f"SD = {std_ratio:.3f}")
print(f"SEM = {sem_ratio:.3f}")

In [ ]:
# === Jitter plot: FRL/WL particle ratio per tomogram ===

import numpy as np
import matplotlib.pyplot as plt

plot_df = counts.reset_index().copy()
plot_df = plot_df[np.isfinite(plot_df["FRL_WL_ratio"])].copy()

# One x-category with horizontal jitter
rng = np.random.default_rng(42)
plot_df["x_jitter"] = rng.normal(loc=0, scale=0.045, size=len(plot_df))

median_ratio = plot_df["FRL_WL_ratio"].median()
std_ratio  = plot_df["FRL_WL_ratio"].std()
sem_ratio  = plot_df["FRL_WL_ratio"].sem()
n_tomos    = len(plot_df)

fig, ax = plt.subplots(figsize=(3.2, 4.5))

ax.scatter(
    plot_df["x_jitter"],
    plot_df["FRL_WL_ratio"],
    s=65,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.6,
)

# Mean ± SD
ax.errorbar(
    0,
    median_ratio,
    yerr=std_ratio,
    fmt="none",
    capsize=8,
    linewidth=1.5,
    color="black",
)

ax.hlines(
    mean_ratio,
    -0.16,
    0.16,
    linewidth=2.0,
    color="black",
)

ax.set_xlim(-0.35, 0.35)
ax.set_xticks([0])
ax.set_xticklabels([f"Tomograms\nn={n_tomos}"])

ax.set_ylabel("Fraction of WL particles")
ax.set_title("Fraction of WL particles per tomogram")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

print(f"n tomograms = {n_tomos}")
print(f"median WL fraction = {median_ratio:.3f}")
print(f"SD = {std_ratio:.3f}")
print(f"SEM = {sem_ratio:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(3.2, 4.5))

ax.hist(
    plot_df["FRL_WL_ratio"],
    alpha=0.85,
    edgecolor="black",
    linewidth=1,
    bins=10
)

plt.show()